# RAG 튜토리얼

# 0. API Key 로드 

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

# 1. RAG system 빠른 구현 (전체 흐름 파악 위함)

- 은행 대출 상품 설명서를 RAG를 통해 모델에 전달하여 고객 질문에 자동 답변하도록 시스템을 구현해보자 

In [2]:
#!pip install langchain_community langchain_text_splitters pymupdf chromadb

In [2]:
from langchain_community.document_loaders import PyMuPDFLoader  # pdf파일 읽을수 있는 클래스
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

In [5]:
# 1. 데이터 로드 
data_path = "../data/하나은행_전세자금대출_상품설명서.pdf"
docs = PyMuPDFLoader(data_path).load()
# docs

# 2. 문서 분할 (chunking)
splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=100)
split_docs = splitter.split_documents(docs)

# 3. 임베딩 객체 정의 
embedding = OpenAIEmbeddings(model="text-embedding-3-small")

# 4. vectorstore 객체 정의 
# vectorstore = Chroma.from_documents(
#     documents=split_docs,
#     embedding=embedding,
#     persist_directory="./Chroma",
#     collection_name="sesac1"
# )

# 5. retrieve(검색 진행)
input_data = "중도상환 수수료는 얼마인가요?"
retriever = vectorstore.as_retriever()
result = retriever.invoke(input_data)


NameError: name 'vectorstore' is not defined

In [ ]:
print("result 자료형:", type(result))
print("반환 문서 개수:", len(result))
print("첫 번째 문서 확인:\n", result[2])

result 자료형: <class 'list'>
반환 문서 개수: 4
첫 번째 문서 확인:
 page_content='☞ 최초 대출취급일로부터 3년까지 적용합니다. 대출만기까지 3개월 미만이 남은 경우에는 중도상환해약금이 부과되지 않습니다.
※ 중도상환해약금이란 대출 상환기일이 도래하기 전에 대출금을 상환할 경우 고객이 부담하는 금액입니다.
¶다만, 기존 대출 계약을 해지하고 동일 은행과 사실상 동일한 계약(기존 계약에 따라 지급된 금전 등을 상환받는 새로운 계약)을 
체결한 경우, 양 계약의 유지기간을 합하여 3년이 경과한 후 해지할 경우에는 중도상환해약금이 면제됩니다.
¶(예시) 1년 만기 대출을 받고나서 6개월 후 대출금 1억원을 상환할 경우 나에게 적용되는 중도상환해약금은?
1억원 x 0.7% x (182 ÷ 365) = 349,041원 (윤년의 경우 348,087원)
□✓ 인지세 : (                                    ) 원
※ 인지세란 인지세법에 의해 대출약정 체결시 납부하는 세금으로 대출금액에 따라 세액이 차등 적용됩니다.
대출금액
5천만원 이하
5천만원 초과
1억원 이하
1억원 초과
10억원 이하
10억원 초과
인지세액
비과세
7만원
15만원
35만원
고객부담
-
3만 5천원
7만 5천원
17만 5천원
은행부담
-' metadata={'trapped': '', 'subject': '', 'file_path': '../../data/하나은행_전세자금대출_상품설명서.pdf', 'modDate': "D:20241227104905+09'00'", 'creationDate': "D:20241227104901+09'00'", 'total_pages': 26, 'creationdate': '2024-12-27T10:49:01+09:00', 'page': 2, 'title': '', 'moddate': '2024-12-27T10:49:05+09:00', 'author': '', 'producer': 'Adobe PDF Lib

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

system_prompt = """\
당신은 하나은행 대출 전문 상담사입니다. 아래 제공된 상품 설명서 내용을 바탕으로 고객의 질문에 정확하게 답변하세요.

답변 원칙:
1. 문서에 있는 정보만 사용하세요.
2. 금리, 수수료 등 구체적 수치를 명시하세요.
3. 정보가 없으면 "해당 정보는 상품설명서에 없습니다. 1599-8000으로 문의하세요"라고 답하세요.
4. 전문용어는 쉽게 풀어서 설명하세요.
5. 고객에게 불리한 조건(연체이자, 기한이익상실 등)은 반드시 안내하세요.\
""" 

user_prompt ="""\
# 고객 질문
{input_data}

# 상품설명서
{context}
"""

template = ChatPromptTemplate([
    ("system", system_prompt),
    ("user", user_prompt)
])
model = ChatOpenAI(model="gpt-5-nano", temperature=0)
chain = template | model | StrOutputParser()


input_data = "중도상환 수수료는 얼마인가요?"
context = "\n\n".join([doc.page_content for doc in result])  # 본문 내용만 전달할 경우 context를 전달
response = chain.invoke({"input_data": input_data, "context": result})
print(response)


다음은 상품설명서에 기재된 중도상환해약금(중도상환 수수료) 관련 내용입니다.

- 적용 기간: 최초 대출취급일로부터 3년까지 적용됩니다.
- 면제 조건: 대출 만기까지 남은 기간이 3개월 미만인 경우에는 중도상환해약금이 부과되지 않습니다.
- 산정 방식: 대출금액 × 0.7% × (경과일수 ÷ 365)
  - 예시: 대출금액 1억원(=100,000,000원)이고 6개월(약 182일) 경과 후 상환하면
    100,000,000 × 0.007 × (182 ÷ 365) = 약 349,041원
    (윤년의 경우 약 348,087원)
- 특별 규정: 기존 대출 계약을 해지하고 동일 은행과 사실상 동일한 계약으로 재체결한 경우, 두 계약의 유지기간을 합산해 3년이 경과한 뒤 해지하면 중도상환해약금이 면제될 수 있습니다.
- 참고: 문서에는 중도상환해약금의 부과 여부를 “대상/비대상”으로 표기하지만, 본 상품의 규정은 3년 적용 및 면제 조건을 함께 안내하고 있습니다.

요약하면, 중도상환해약금은 대출금액의 0.7%에 해당하는 금액을 경과일수 비율로 산정하며, 3년 이내에 상환하는 경우에만 적용되고 3개월 남으면 면제됩니다. 구체 금액은 대출금액과 실제 경과일수에 따라 달라집니다.


# 2. 단계별 상세 설명

## 2.1 문서 로드 (Document Loader)

- 다양한 형식의 문서를 LangChain의 Document 객체로 변환하는 단계
- 현재 실습에서는 PDF 형식의 문서를 Document 객체로 변환하고자 함

In [1]:
from langchain_community.document_loaders import PyMuPDFLoader

data_path = "../data/하나은행_전세자금대출_상품설명서.pdf"
docs = PyMuPDFLoader(data_path, extract_tables="markdown").load()  # extract_tables="markdown" pdf파일에 있는 표를 markdown형식으로 뽑아줌. 
                                                                   # 이걸 설정안하면 str만 출력되어 기존에 테이블 형식이었는지 알 수 없음
print(type(docs))
print(len(docs))
docs[0]


Consider using the pymupdf_layout package for a greatly improved page layout analysis.
<class 'list'>
26


Document(metadata={'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 19.4 (Windows)', 'creationdate': '2024-12-27T10:49:01+09:00', 'source': '../data/하나은행_전세자금대출_상품설명서.pdf', 'file_path': '../data/하나은행_전세자금대출_상품설명서.pdf', 'total_pages': 26, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2024-12-27T10:49:05+09:00', 'trapped': '', 'modDate': "D:20241227104905+09'00'", 'creationDate': "D:20241227104901+09'00'", 'page': 0}, page_content='5-06-0900(26-1) (2025.01 개정)\n(보존년한 : 완제일로부터 10년)\n준법감시인 심의필 제 2024-설명서-280 호\n(은행보관용)\t\n전세자금대출 상품설명서\n- 고객님께서는 상품 가입 전 아래 사항을 반드시 확인 · 숙지하여 주시기 바랍니다 -\n본인확인\n담당\n책임자\n지점장\n◈ \x07이 설명서는 금융소비자의 권익 보호 및 대출상품에 대한 이해 증진을 위하여 「금융소비자 보호에 관한 법률」 및 관련 규정에 \n의거, 은행의 내부 통제절차를 거쳐 대출상품의 주요 내용을 쉽게 이해할 수 있도록 작성한 자료입니다.\n◈ \x07설명내용을 제대로 이해하지 못하였음에도 불구하고 설명을 이해했다는 서명을 하거나 녹취기록을 남기시는 경우, 추후 해당 \n내용과 관련한 권리구제가 어려울 수 있습니다.\n유사 상품과 구별되는 특징\n¶\x07 \x07전세자금대출은 세입자(임차인)가 임대인으로부터 전세금(임차금)을 돌려받을 권리(임차보증금반환채권)를 담보로 하여\n(질권설정

In [2]:
# Document 객체의 메타데이터 확인
docs[0].metadata  # metadata {} 딕셔너리 형태로 출력됨.

{'producer': 'Adobe PDF Library 17.0',
 'creator': 'Adobe InDesign 19.4 (Windows)',
 'creationdate': '2024-12-27T10:49:01+09:00',
 'source': '../data/하나은행_전세자금대출_상품설명서.pdf',
 'file_path': '../data/하나은행_전세자금대출_상품설명서.pdf',
 'total_pages': 26,
 'format': 'PDF 1.4',
 'title': '',
 'author': '',
 'subject': '',
 'keywords': '',
 'moddate': '2024-12-27T10:49:05+09:00',
 'trapped': '',
 'modDate': "D:20241227104905+09'00'",
 'creationDate': "D:20241227104901+09'00'",
 'page': 0}

In [3]:
# 페이지 본문 확인
print(docs[0].page_content)

5-06-0900(26-1) (2025.01 개정)
(보존년한 : 완제일로부터 10년)
준법감시인 심의필 제 2024-설명서-280 호
(은행보관용)	
전세자금대출 상품설명서
- 고객님께서는 상품 가입 전 아래 사항을 반드시 확인 · 숙지하여 주시기 바랍니다 -
본인확인
담당
책임자
지점장
◈ 이 설명서는 금융소비자의 권익 보호 및 대출상품에 대한 이해 증진을 위하여 「금융소비자 보호에 관한 법률」 및 관련 규정에 
의거, 은행의 내부 통제절차를 거쳐 대출상품의 주요 내용을 쉽게 이해할 수 있도록 작성한 자료입니다.
◈ 설명내용을 제대로 이해하지 못하였음에도 불구하고 설명을 이해했다는 서명을 하거나 녹취기록을 남기시는 경우, 추후 해당 
내용과 관련한 권리구제가 어려울 수 있습니다.
유사 상품과 구별되는 특징
¶ 전세자금대출은 세입자(임차인)가 임대인으로부터 전세금(임차금)을 돌려받을 권리(임차보증금반환채권)를 담보로 하여
(질권설정·채권양도) 세입자에게 취급하는 대출상품입니다.
    예시 : 전세금안심대출, 우량전세자금대출등
¶ 일반적으로 전세자금대출은 금리수준이 신용대출보다 낮고 주택담보대출과 유사하나, 연체 등이 발생하는 경우 임차보증금
에 대한 손실이 발생할 수 있으며, 대출 담보를 위해 임차보증금에 질권 설정 시 해당 사실에 대한 임대인 통보 또는 임대인
의 승낙 등 협조가 필요할 수 있습니다.
민원 · 상담이 빈번하여 숙지가 필요한 사항
Q1. 전세자금대출 최초 취급 시 우대금리를 적용받은 경우, 연장시점에도 유지가 가능한가요?
A. 대출 연장시점에서 은행의 우대금리 요건을 갖춘 경우 가능하나, 연장시점에서 고객님의 조건 충족여부가 달라졌을 수 있으
므로 대출연장 전에 은행에 문의하시기 바랍니다.
Q2. 전세자금대출 연장 시 일정금액 이상 상환해야 하나요?
A. 상품별로 기한연장 시 일정금액 이상 분할상환 약정이 필수이므로, 해당 내용을 유의하시어 연장처리 하시기 바랍니다.
Q3. 전세계약 만료 시 전세금 반환 또는 연장 관련하여 숙지

In [4]:
# 페이지별로 몇 글자로 구성돼있는지 확인
for page in docs:
    print(len(page.page_content)) # 장표가 존재하면 해당 장표는 제외시켜주는 게 좋음(의미있는 내용이 아니니까!)

2140
1123
2320
2532
2401
3727
2250
1835
2093
2032
2791
2654
3062
2081
1124
2321
2533
2402
3728
2251
1836
2094
2032
2791
2654
3062


In [5]:
# 은행별로 메타데이터에 은행 명시

docs_hana = PyMuPDFLoader("../data/하나은행_전세자금대출_상품설명서.pdf", extract_tables="markdown").load()
for doc in docs_hana:
    doc.metadata["bank"] = "하나은행"

docs_kb = PyMuPDFLoader("../data/국민은행_전세자금대출_상품설명서.pdf", extract_tables="markdown").load()
for doc in docs_kb:
    doc.metadata["bank"] = "국민은행"

# 확인
display(docs_hana[0].metadata)
display(docs_kb[0].metadata)


{'producer': 'Adobe PDF Library 17.0',
 'creator': 'Adobe InDesign 19.4 (Windows)',
 'creationdate': '2024-12-27T10:49:01+09:00',
 'source': '../data/하나은행_전세자금대출_상품설명서.pdf',
 'file_path': '../data/하나은행_전세자금대출_상품설명서.pdf',
 'total_pages': 26,
 'format': 'PDF 1.4',
 'title': '',
 'author': '',
 'subject': '',
 'keywords': '',
 'moddate': '2024-12-27T10:49:05+09:00',
 'trapped': '',
 'modDate': "D:20241227104905+09'00'",
 'creationDate': "D:20241227104901+09'00'",
 'page': 0,
 'bank': '하나은행'}

{'producer': 'Adobe PDF Library 9.0',
 'creator': 'Adobe InDesign CS4 (6.0)',
 'creationdate': '2024-10-14T13:04:05+09:00',
 'source': '../data/국민은행_전세자금대출_상품설명서.pdf',
 'file_path': '../data/국민은행_전세자금대출_상품설명서.pdf',
 'total_pages': 20,
 'format': 'PDF 1.4',
 'title': '',
 'author': '',
 'subject': '',
 'keywords': '',
 'moddate': '2024-10-14T13:04:13+09:00',
 'trapped': '',
 'modDate': "D:20241014130413+09'00'",
 'creationDate': "D:20241014130405+09'00'",
 'page': 0,
 'bank': '국민은행'}

In [6]:
# document 하나로 합치기
docs = docs_hana + docs_kb

print(len(docs))
print(docs[0].metadata["bank"])
print(docs[-1].metadata["bank"])

46
하나은행
국민은행


## 2.2 Split Document(문서 분할)

- 문서를 분할하는 이유
    - 긴 문서는 임베딩 시 토큰 제한 초과에 걸림
    - 관련 없는 정보까지 포함되면 답변 품질 저하
    - 적절한 크기로 나누면 검색 정확도를 향상시킬 수 있음

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# splitter 객체 정의
splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=100) #600자 내에 100글자 포함되어 있는 거당

# 문서 분할
split_docs = splitter.split_documents(docs)

# 확인
print(type(split_docs))                 # 데이터 타입 확인
print(len(split_docs))                  # split 후 총 청킹된 document 개수 확인
print()
print(len(split_docs[0].page_content))  # 특정 페이지의 글자 수 확인(대략 600글자인지 확인)
print(split_docs[0].page_content)       # split된 청크 본문 확인
print()
split_docs[0]                           # 특정 페이지의 데이터 형태 확인(메타데이터가 유지돼있음을 알 수 있음)

<class 'list'>
446

597
5-06-0900(26-1) (2025.01 개정)
(보존년한 : 완제일로부터 10년)
준법감시인 심의필 제 2024-설명서-280 호
(은행보관용)	
전세자금대출 상품설명서
- 고객님께서는 상품 가입 전 아래 사항을 반드시 확인 · 숙지하여 주시기 바랍니다 -
본인확인
담당
책임자
지점장
◈ 이 설명서는 금융소비자의 권익 보호 및 대출상품에 대한 이해 증진을 위하여 「금융소비자 보호에 관한 법률」 및 관련 규정에 
의거, 은행의 내부 통제절차를 거쳐 대출상품의 주요 내용을 쉽게 이해할 수 있도록 작성한 자료입니다.
◈ 설명내용을 제대로 이해하지 못하였음에도 불구하고 설명을 이해했다는 서명을 하거나 녹취기록을 남기시는 경우, 추후 해당 
내용과 관련한 권리구제가 어려울 수 있습니다.
유사 상품과 구별되는 특징
¶ 전세자금대출은 세입자(임차인)가 임대인으로부터 전세금(임차금)을 돌려받을 권리(임차보증금반환채권)를 담보로 하여
(질권설정·채권양도) 세입자에게 취급하는 대출상품입니다.
    예시 : 전세금안심대출, 우량전세자금대출등
¶ 일반적으로 전세자금대출은 금리수준이 신용대출보다 낮고 주택담보대출과 유사하나, 연체 등이 발생하는 경우 임차보증금



Document(metadata={'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 19.4 (Windows)', 'creationdate': '2024-12-27T10:49:01+09:00', 'source': '../data/하나은행_전세자금대출_상품설명서.pdf', 'file_path': '../data/하나은행_전세자금대출_상품설명서.pdf', 'total_pages': 26, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2024-12-27T10:49:05+09:00', 'trapped': '', 'modDate': "D:20241227104905+09'00'", 'creationDate': "D:20241227104901+09'00'", 'page': 0, 'bank': '하나은행'}, page_content='5-06-0900(26-1) (2025.01 개정)\n(보존년한 : 완제일로부터 10년)\n준법감시인 심의필 제 2024-설명서-280 호\n(은행보관용)\t\n전세자금대출 상품설명서\n- 고객님께서는 상품 가입 전 아래 사항을 반드시 확인 · 숙지하여 주시기 바랍니다 -\n본인확인\n담당\n책임자\n지점장\n◈ \x07이 설명서는 금융소비자의 권익 보호 및 대출상품에 대한 이해 증진을 위하여 「금융소비자 보호에 관한 법률」 및 관련 규정에 \n의거, 은행의 내부 통제절차를 거쳐 대출상품의 주요 내용을 쉽게 이해할 수 있도록 작성한 자료입니다.\n◈ \x07설명내용을 제대로 이해하지 못하였음에도 불구하고 설명을 이해했다는 서명을 하거나 녹취기록을 남기시는 경우, 추후 해당 \n내용과 관련한 권리구제가 어려울 수 있습니다.\n유사 상품과 구별되는 특징\n¶\x07 \x07전세자금대출은 세입자(임차인)가 임대인으로부터 전세금(임차금)을 돌려받을 권리(임차보증금반환채권

In [8]:
split_docs[1].metadata  # docs는 분할됐더라도 패이지 메타정보는 그대로 보존돼있음을 확인 

{'producer': 'Adobe PDF Library 17.0',
 'creator': 'Adobe InDesign 19.4 (Windows)',
 'creationdate': '2024-12-27T10:49:01+09:00',
 'source': '../data/하나은행_전세자금대출_상품설명서.pdf',
 'file_path': '../data/하나은행_전세자금대출_상품설명서.pdf',
 'total_pages': 26,
 'format': 'PDF 1.4',
 'title': '',
 'author': '',
 'subject': '',
 'keywords': '',
 'moddate': '2024-12-27T10:49:05+09:00',
 'trapped': '',
 'modDate': "D:20241227104905+09'00'",
 'creationDate': "D:20241227104901+09'00'",
 'page': 0,
 'bank': '하나은행'}

In [9]:
# 청크별 글자 수 확인
for page in split_docs:
    print(len(page.page_content))

597
575
567
599
41
592
595
561
572
594
194
379
255
575
580
579
594
406
109
569
547
585
386
543
592
584
547
316
55
595
599
178
557
61
591
596
554
40
599
119
599
574
175
484
191
585
583
593
445
101
546
528
572
589
126
574
598
300
570
318
598
134
566
591
331
442
547
410
579
593
536
477
476
556
582
575
567
599
592
596
561
572
594
195
379
255
575
580
579
594
407
109
569
547
585
387
543
593
584
547
316
55
595
599
178
557
61
591
596
555
40
599
119
599
574
176
484
191
585
583
593
446
101
546
528
572
589
126
574
598
300
570
318
598
134
566
591
331
442
547
410
579
593
536
477
476
556
579
593
593
438
596
511
537
566
568
594
357
423
550
571
534
596
538
194
598
591
588
573
187
548
321
397
588
53
581
577
598
553
502
529
220
593
551
596
554
561
99
537
588
531
558
574
290
82
554
589
597
538
539
560
252
598
585
595
596
599
445
89
592
594
599
594
596
585
595
593
599
598
594
591
593
590
599
599
597
599
594
599
598
596
596
594
588
596
597
597
596
585
592
580
586
587
596
598
599
598
598
596
596
596
597
596

## 2.3 임베딩 객체 정의 

- 차원이 높을 수록 정교한 표현 가능 (절대적이지 않고 문서마다 적절하게 잘 표현되는 차원을 찾아야함)
- 어떤 임베딩 알고리즘을 쓰느냐에 따라 사용자 입력값과 유사도를 계산했을 때 유사도가 다르게 나올 수 있음 
- OpenAI 임베딩 모델
    - 다국어 지원이 되는 모델 중 가장 성능이 무난해서 많이 쓰임 
    - 좋은 모델 같은 경우는 모델 사이즈가 커서 GPU환경에서 돌려야할 수도 있음, 
        - OpenAI의 임베딩 모델을 쓰는 경우엔 OpenAI의 자원을 빌려쓰기 때문에 무거운 모델도 일반 노트북에서 돌릴 수 있음 
    - 하지만 비용 발생 
    - https://platform.openai.com/docs/guides/embeddings#embedding-models

- 임베딩이란?
    - 텍스트를 숫자 벡터로 변환
    - 의미가 비슷한 텍스트는 벡터 공간에서 가까이 위치
    - 벡터 거리로 유사도 계산

In [10]:
from langchain_openai import OpenAIEmbeddings

# 임베딩 객체 정의
embedding = OpenAIEmbeddings(model="text-embedding-3-small")

# 확인용(임베딩 후 결과 확인)
embedded_document = embedding.embed_documents([split_docs[0].page_content]) # 리스트 안에 넣어야 한 개의 청크가 하나의 벡터로 임베딩됨

print("원문 데이터 타입: ", type(split_docs[0].page_content), "\n")
print("embedded_document 데이터 타입: ", type(embedded_document), "\n") # list 
print("첫 번째 요소의 데이터 타입: ", type(embedded_document[0]), "\n") # list 
print("embedded_document의 요소 개수: ", len(embedded_document), "\n") # 임베딩된 청크 개수 
print("embedded_document의 첫 번째 요소 길이: ", len(embedded_document[0]), "\n") # 총 사전 개수 (임베딩 차원, 해당 모델은 총 1536차원)
print("첫 번째 토큰의 벡터값 확인:\n", embedded_document[0][:10]) # 앞 10개 정도만 확인

원문 데이터 타입:  <class 'str'> 

embedded_document 데이터 타입:  <class 'list'> 

첫 번째 요소의 데이터 타입:  <class 'list'> 

embedded_document의 요소 개수:  1 

embedded_document의 첫 번째 요소 길이:  1536 

첫 번째 토큰의 벡터값 확인:
 [-0.012029633857309818, 0.0266428142786026, -0.004450561013072729, 0.03368701413273811, -0.011636046692728996, 0.023029886186122894, -0.008598363026976585, 0.039358701556921005, 0.018690338358283043, -0.022444551810622215]


## 2.4 VectorStore 정의

- Vector Store의 역할:
    1. 임베딩 벡터를 저장
    2. 질문 벡터와 유사한 문서 벡터 검색
    3. 빠른 검색을 위한 인덱싱

- Chroma
    - 대표적인 로컬 DB
    - 오픈소스 벡터 데이터베이스
    - 임베딩을 저장하고 검색할 수 있도록 설계된 데이터베이스
    - 벡터 검색 및 정보 검색과 같은 작업에 사용 됨

In [ ]:
from langchain_community.vectorstores import Chroma
# 아래의 코드는 제일 처음 한 번만 실행시켜주기! 
# vectorstore = Chroma.from_documents(
#    documents=split_docs,
#    embedding=embedding,
#    persist_directory="./Chroma",
#    collection_name="sesac2"
# )

# 기존에 존재하는 collection을 불러올 때는 아래의 코드를 통해 객체를 다시 생성해주기
vectorstore = Chroma(
    embedding_function=embedding,
    persist_directory="./Chroma",
    collection_name="sesac2"
)

print(vectorstore._collection.count())

446


In [17]:
# 각 정보 확인해보기

store_info = vectorstore.get() # .get(): vectorstore에 저장된 청크 ID 및 임베딩 값, 원문 등의 정보 확인 {}로 반환.
store_info

{'ids': ['c8913c5f-92d7-4fdc-a536-e22d4068929b',
  '0f8771b4-2293-4ed6-8b54-9565c9d64f5f',
  'f9346ecd-0d43-455c-a73e-9a7662edf355',
  '9e6afd73-67d6-4524-a8d5-1ed51f33b7af',
  'f58941db-a762-456a-aab3-91e102727972',
  'bc208e05-9745-4e2d-a6fa-174c57fb30f0',
  '6d43bdd5-a313-4d03-adfa-90321f76c019',
  '7ac0b432-abad-4ba1-a797-907c5e8fadbf',
  '1da6c56e-f506-444d-b08b-98b9a3e08caa',
  'aa79cb9b-002a-4f37-96ed-3959e97b63d0',
  'c40a9c85-ee81-4002-a53d-84998a7f1b24',
  'b805b782-bc6a-44a3-82c3-4927760c73c2',
  'd2008767-cc43-4c01-9430-bdf8fa53017f',
  'bd0f355d-9f3c-46c4-9324-fe552c7b079f',
  '54ea3806-5665-4693-98ae-627cf8c54809',
  'ef272716-96bb-4f98-8c98-41a0cda370e0',
  '319eb59c-1f4b-4992-9473-1efca3265a8e',
  '5077695f-225d-476b-9745-cfefaefc08d6',
  '3b23a604-5f71-4e0d-92d0-ce4b6d9551fc',
  '943e8895-c4d0-4d55-a216-c566d503118c',
  '720ebe34-8dbd-44f7-9ebc-dccea2b59f21',
  '1659b1aa-0aff-4f7d-8ce3-fa24ba518c5b',
  '5fc1263a-e8ff-4770-8b73-5035635f0b11',
  '1e087aec-d5ac-43d8-a790-

In [18]:
print(type(store_info))
print(store_info.keys())
print(len(store_info["ids"]))

<class 'dict'>
dict_keys(['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas'])
446


In [19]:
# 벡터값은 비어있음 ->  embeddings는 기본적으로 생략되어 반환되지 않음.
store_info["embeddings"] 

In [20]:
# 벡터값 확인하기 위해서는 별도로 벡터값을 불러오도록 지정 피룡
store_info = vectorstore.get(include=["embeddings"])

print(len(store_info["embeddings"]))
store_info["embeddings"]

446


array([[-0.01208996,  0.02674324, -0.00441516, ...,  0.01000601,
        -0.0189221 , -0.00033871],
       [ 0.01287922,  0.05685209,  0.02428592, ..., -0.02079668,
        -0.01383956,  0.01979366],
       [ 0.01200715,  0.04348249,  0.03504276, ..., -0.01168092,
         0.00769783, -0.00679809],
       ...,
       [ 0.03291217,  0.0525584 ,  0.07989046, ..., -0.02998524,
        -0.01342387, -0.00120091],
       [ 0.0259121 , -0.00185809,  0.03329389, ...,  0.00350129,
         0.00299253, -0.00297989],
       [ 0.0291982 ,  0.05873815,  0.04728955, ..., -0.01235637,
         0.00566022, -0.00491265]], shape=(446, 1536))

In [21]:
# 문서 추가

# 1. 추가할 문서 로드
data_path = "../data/신한은행_전세자금대출_상품설명서.pdf"
additional_docs = PyMuPDFLoader(data_path).load()  
print("원본 문서 페이지 수:", len(additional_docs))

# 2. split(문서 분할)
split_additional_docs = splitter.split_documents(additional_docs)
print("청킹 후 페이지 수:", len(split_additional_docs))

# 3. vectorstore에 추가문서 저장
vectorstore.add_documents(documents=split_additional_docs)
print("저장한 총 청크 개수:", vectorstore._collection.count())

# 4. store_info 갱신
store_info = vectorstore.get()

원본 문서 페이지 수: 23
청킹 후 페이지 수: 101
저장한 총 청크 개수: 547


In [ ]:
# 특정 청크 삭제 (마지막에 추가된 청크를 삭제해보자)
# vectorstore.delete(ids=store_info["ids"][-101:])  # 추가된 101페이지 삭제
# 삭제 후에는 코드 반드시 주석처리! 만약 아래코드에서 오류나서 다시 돌릴때 또 삭제될 수 있으니까~

# 삭제 후 청크 개수 확인
vectorstore._collection.count()


446

In [ ]:
# collection의 모든 데이터 삭제
# vectorstore.delete(ids=store_info["ids"])
# vectorstore._collection.count()

0

## 2.5 Retriever

- as_retriever(): 
    - 벡터 DB객체를 효율적인 검색기로 사용할 수 있도록 변환해주는 역할 
    - as_retirever를 사용하면 쿼리 최적화 기법들이 적용되고 langchain 생태계와의 연동성이 높아지며 유사도 검색 전후에 필터나 제약조건 등을 추가로 적용할 수 있어서 일반적인 검색기보다 더 성능이 좋음 
    - 추후 langchain의 RetrievalQA 클래스와 연동하여 질의-응답에 대한 전체적인 RAG chaind을 구성할 수 있음 
    - 파라미터
        - search_type: str, 검색 유형 정의 (default=similarity, mmr, similarity_score_threshold)
        - search_kwargs: dict, 검색 함수에 전달할 추가 인자
            - k: 반환할 문서 수 (default: 4)
            - score_threshold: 최소 유사도 임계값
                - 쿼리가 문서와 얼마나 유사한지의 정도
                - 1이면 완벽하게 일치함을 의미
            - fetch_k: MMR 알고리즘에 전달할 문서 수 (default:20)
                - 20이면 먼저 20개를 가져온 후 mmr로 다양성 및 유사도를 계산 후 순위가 높은 개수(k값)를 골라 옴
            - lambda_mult: MMR 결과의 다양성 조절 (default: 0.5, 0~1)
                - 1에 가까운 숫자면 다양성을 고려하여 다양한 문서 반환
                - 0에 가까우면 유사도만 고려하여 반환
            - filter: 문서 메타데이터 필터링

### 2.5.1 기본값 적용

-similarity기반 검색

In [24]:
# 검색기 객체 정의
retriever = vectorstore.as_retriever()

# 사용자 질의로 검색 진행
input_data = "중도상환 수수료가 얼마인가요?"
contexts = retriever.invoke(input_data)

print(type(contexts))
print(len(contexts)) # 기본적으로 4개 반환(가장유사한 순서대로 반환해줌.)
print(contexts[0])


<class 'list'>
4
page_content='E009–264(210×297) 모조지70g/㎡ (2024. 10 개정) K11
(3/10)
고  객  용
[준법감시인 심의필 제2024-4907-2호(유효기간 : 2024.10.17 ~ 2026.09.30)]
1
수수료 등 비용부담 사항
	중도상환수수료 : 중도상환대출금액 × 중도상환수수료율(%) × (대출잔여일수 ÷ 대출기간)
	
☞ 중도상환수수료율은 상품에 따라 0.6%~0.7%입니다. (통상 고정금리 및 혼합금리 방식이 변동금리 방식보다 0.1%p 높습니다.)
	
☞ 최초 대출취급일로부터 3년까지 적용합니다. 대출만기 도래 전 3개월 이내에 상환하는 경우에는 중도상환수수료가 부과되지 
않습니다.
	
※ 중도상환수수료란 대출의 상환기일이 도래하기 전에 대출금을 상환할 경우 고객이 부담하는 금액입니다.
	
• 다만, 기존 대출 계약을 해지하고 동일 은행과 사실상 동일한 계약(기존 계약에 따라 지급된 금전 등을 상환받는 새로운 계약)을 체
결한 경우, 양 계약의 유지기간을 합하여 3년이 경과한 후 해지할 경우에는 중도상환수수료가 면제됩니다.' metadata={'file_path': '../data/국민은행_전세자금대출_상품설명서.pdf', 'creationdate': '2024-10-14T13:04:05+09:00', 'modDate': "D:20241014130413+09'00'", 'creator': 'Adobe InDesign CS4 (6.0)', 'trapped': '', 'title': '', 'subject': '', 'keywords': '', 'page': 12, 'source': '../data/국민은행_전세자금대출_상품설명서.pdf', 'producer': 'Adobe PDF Library 9.0', 'moddate': '2024-10-14T13:04:13+09:00', 'total_pages': 20, 'author': '', 'bank': '국민은행', 'creationDat

In [25]:
for context in contexts:
    print(context.page_content)
    print("\n", "-"*70)

E009–264(210×297) 모조지70g/㎡ (2024. 10 개정) K11
(3/10)
고  객  용
[준법감시인 심의필 제2024-4907-2호(유효기간 : 2024.10.17 ~ 2026.09.30)]
1
수수료 등 비용부담 사항
	중도상환수수료 : 중도상환대출금액 × 중도상환수수료율(%) × (대출잔여일수 ÷ 대출기간)
	
☞ 중도상환수수료율은 상품에 따라 0.6%~0.7%입니다. (통상 고정금리 및 혼합금리 방식이 변동금리 방식보다 0.1%p 높습니다.)
	
☞ 최초 대출취급일로부터 3년까지 적용합니다. 대출만기 도래 전 3개월 이내에 상환하는 경우에는 중도상환수수료가 부과되지 
않습니다.
	
※ 중도상환수수료란 대출의 상환기일이 도래하기 전에 대출금을 상환할 경우 고객이 부담하는 금액입니다.
	
• 다만, 기존 대출 계약을 해지하고 동일 은행과 사실상 동일한 계약(기존 계약에 따라 지급된 금전 등을 상환받는 새로운 계약)을 체
결한 경우, 양 계약의 유지기간을 합하여 3년이 경과한 후 해지할 경우에는 중도상환수수료가 면제됩니다.

 ----------------------------------------------------------------------
E009–264(210×297) 모조지70g/㎡ (2024. 10 개정) K11
(3/10)
은  행  용
[준법감시인 심의필 제2024-4907-2호(유효기간 : 2024.10.17 ~ 2026.09.30)]
1
수수료 등 비용부담 사항
	중도상환수수료 : 중도상환대출금액 × 중도상환수수료율(%) × (대출잔여일수 ÷ 대출기간)
	
☞ 중도상환수수료율은 상품에 따라 0.6%~0.7%입니다. (통상 고정금리 및 혼합금리 방식이 변동금리 방식보다 0.1%p 높습니다.)
	
☞ 최초 대출취급일로부터 3년까지 적용합니다. 대출만기 도래 전 3개월 이내에 상환하는 경우에는 중도상환수수료가 부과되지 
않습니다.
	
※ 중도상환수수료란 대출의 상환기일이 도래하기 전에 대출금을 상환할 경우 고객이

### 2.5.2 옵션 변경

- mmr 기반 검색

In [26]:
retriever_mmr = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 6,            # 최종 반환할 문서 수
        "fetch_k": 10,     # 우선적으로 선발할 문서 수 (먼저 10개 선별 후 다양성 및 유사도 계산됨)
        "lambda_mult": 0.8 # 다양성 조절(0~1, 1에 가까울수록 중복없이 다양한 문서 반환)

    }
)

contexts_mmr = retriever_mmr.invoke(input_data)
print(type(contexts_mmr))
print(len(contexts_mmr)) # 기본적으로 4개 반환(가장유사한 순서대로 반환해줌.)
print(contexts_mmr[0])

<class 'list'>
6
page_content='E009–264(210×297) 모조지70g/㎡ (2024. 10 개정) K11
(3/10)
고  객  용
[준법감시인 심의필 제2024-4907-2호(유효기간 : 2024.10.17 ~ 2026.09.30)]
1
수수료 등 비용부담 사항
	중도상환수수료 : 중도상환대출금액 × 중도상환수수료율(%) × (대출잔여일수 ÷ 대출기간)
	
☞ 중도상환수수료율은 상품에 따라 0.6%~0.7%입니다. (통상 고정금리 및 혼합금리 방식이 변동금리 방식보다 0.1%p 높습니다.)
	
☞ 최초 대출취급일로부터 3년까지 적용합니다. 대출만기 도래 전 3개월 이내에 상환하는 경우에는 중도상환수수료가 부과되지 
않습니다.
	
※ 중도상환수수료란 대출의 상환기일이 도래하기 전에 대출금을 상환할 경우 고객이 부담하는 금액입니다.
	
• 다만, 기존 대출 계약을 해지하고 동일 은행과 사실상 동일한 계약(기존 계약에 따라 지급된 금전 등을 상환받는 새로운 계약)을 체
결한 경우, 양 계약의 유지기간을 합하여 3년이 경과한 후 해지할 경우에는 중도상환수수료가 면제됩니다.' metadata={'creationdate': '2024-10-14T13:04:05+09:00', 'producer': 'Adobe PDF Library 9.0', 'keywords': '', 'page': 12, 'creator': 'Adobe InDesign CS4 (6.0)', 'file_path': '../data/국민은행_전세자금대출_상품설명서.pdf', 'title': '', 'subject': '', 'source': '../data/국민은행_전세자금대출_상품설명서.pdf', 'bank': '국민은행', 'format': 'PDF 1.4', 'trapped': '', 'moddate': '2024-10-14T13:04:13+09:00', 'modDate': "D:20241014130413+09'00'", 'total_pages': 20, 'auth

In [27]:
for context in contexts_mmr:
    print(context.page_content)
    print("\n", "-"*70)

E009–264(210×297) 모조지70g/㎡ (2024. 10 개정) K11
(3/10)
고  객  용
[준법감시인 심의필 제2024-4907-2호(유효기간 : 2024.10.17 ~ 2026.09.30)]
1
수수료 등 비용부담 사항
	중도상환수수료 : 중도상환대출금액 × 중도상환수수료율(%) × (대출잔여일수 ÷ 대출기간)
	
☞ 중도상환수수료율은 상품에 따라 0.6%~0.7%입니다. (통상 고정금리 및 혼합금리 방식이 변동금리 방식보다 0.1%p 높습니다.)
	
☞ 최초 대출취급일로부터 3년까지 적용합니다. 대출만기 도래 전 3개월 이내에 상환하는 경우에는 중도상환수수료가 부과되지 
않습니다.
	
※ 중도상환수수료란 대출의 상환기일이 도래하기 전에 대출금을 상환할 경우 고객이 부담하는 금액입니다.
	
• 다만, 기존 대출 계약을 해지하고 동일 은행과 사실상 동일한 계약(기존 계약에 따라 지급된 금전 등을 상환받는 새로운 계약)을 체
결한 경우, 양 계약의 유지기간을 합하여 3년이 경과한 후 해지할 경우에는 중도상환수수료가 면제됩니다.

 ----------------------------------------------------------------------
E009–264(210×297) 모조지70g/㎡ (2024. 10 개정) K11
(3/10)
은  행  용
[준법감시인 심의필 제2024-4907-2호(유효기간 : 2024.10.17 ~ 2026.09.30)]
1
수수료 등 비용부담 사항
	중도상환수수료 : 중도상환대출금액 × 중도상환수수료율(%) × (대출잔여일수 ÷ 대출기간)
	
☞ 중도상환수수료율은 상품에 따라 0.6%~0.7%입니다. (통상 고정금리 및 혼합금리 방식이 변동금리 방식보다 0.1%p 높습니다.)
	
☞ 최초 대출취급일로부터 3년까지 적용합니다. 대출만기 도래 전 3개월 이내에 상환하는 경우에는 중도상환수수료가 부과되지 
않습니다.
	
※ 중도상환수수료란 대출의 상환기일이 도래하기 전에 대출금을 상환할 경우 고객이

## 2.6 최종 프롬프트 완성하기 

In [28]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

In [29]:
system_prompt = """\
당신은 대출 전문 상담사입니다. 아래 제공된 상품설명서 내용을 바탕으로 고객의 질문에 정확하게 답변하세요.

답변 원칙:
1. 어느 은행의 몇페이지 정보인지 언급하세요.
2. 문서에 있는 정보만 사용하세요.
3. 금리, 수수료 등 구체적 수치를 명시하세요.
4. 정보가 없으면 "해당 정보는 상품설명서에 없습니다. 1599-8000으로 문의하세요"라고 답하세요.
5. 전문 용어는 쉽게 풀어서 설명하세요.
6. 고객에게 불리한 조건(연체이자, 기한이익상실 등)은 반드시 안내하세요.\
"""

user_prompt = """
사용자 질의에 대해 아래의 context를 참고하여 답하세요.

--사용자 질의--
{question}

--context--
{context}
"""

final_template = ChatPromptTemplate([
    ("system", system_prompt),
    ("user", user_prompt)
])

model = ChatOpenAI(model="gpt-5-nano", temperature=0)
chain = final_template | model | StrOutputParser()

input_data = "중도상환 수수료는 얼마인가요?"
response = chain.invoke({"question": input_data, "context": contexts})
print(response)

- 은행/문서 및 페이지: 국민은행 전세자금대출 상품설명서, 페이지 2(수수료 등 비용부담 사항 부분)

질문에 대한 답변:
- 중도상환수수료 산정식: 중도상환수수료 = 중도상환대출금액 × 중도상환수수료율(%) × (대출잔여일수 ÷ 대출기간)
- 수수료율 범위: 상품에 따라 0.6% ~ 0.7%입니다. 일반적으로 고정금리 및 혼합금리 방식이 변동금리 방식보다 0.1%p 높습니다.
- 적용 기간: 최초 대출취급일로부터 3년까지 적용됩니다.
- 예외 및 면제 조건:
  - 대출만기 도래 전 3개월 이내에 상환하는 경우에는 중도상환수수료가 부과되지 않습니다.
  - 기존 대출 계약을 해지하고 동일 은행과 사실상 동일한 계약으로 새로운 계약을 체결한 경우, 양 계약의 유지기간을 합하여 3년이 경과한 뒤 해지하면 중도상환수수료가 면제될 수 있습니다.
- 추가 설명: 중도상환수수료란 대출의 상환기일이 도래하기 전에 대출금을 상환할 경우 고객이 부담하는 금액입니다.


In [30]:
for context in contexts:
    if context.metadata["bank"] == "국민은행":
        print("페이지:", context.metadata["page"])
        print(context.page_content)
        print("="*70)

페이지: 12
E009–264(210×297) 모조지70g/㎡ (2024. 10 개정) K11
(3/10)
고  객  용
[준법감시인 심의필 제2024-4907-2호(유효기간 : 2024.10.17 ~ 2026.09.30)]
1
수수료 등 비용부담 사항
	중도상환수수료 : 중도상환대출금액 × 중도상환수수료율(%) × (대출잔여일수 ÷ 대출기간)
	
☞ 중도상환수수료율은 상품에 따라 0.6%~0.7%입니다. (통상 고정금리 및 혼합금리 방식이 변동금리 방식보다 0.1%p 높습니다.)
	
☞ 최초 대출취급일로부터 3년까지 적용합니다. 대출만기 도래 전 3개월 이내에 상환하는 경우에는 중도상환수수료가 부과되지 
않습니다.
	
※ 중도상환수수료란 대출의 상환기일이 도래하기 전에 대출금을 상환할 경우 고객이 부담하는 금액입니다.
	
• 다만, 기존 대출 계약을 해지하고 동일 은행과 사실상 동일한 계약(기존 계약에 따라 지급된 금전 등을 상환받는 새로운 계약)을 체
결한 경우, 양 계약의 유지기간을 합하여 3년이 경과한 후 해지할 경우에는 중도상환수수료가 면제됩니다.
페이지: 2
E009–264(210×297) 모조지70g/㎡ (2024. 10 개정) K11
(3/10)
은  행  용
[준법감시인 심의필 제2024-4907-2호(유효기간 : 2024.10.17 ~ 2026.09.30)]
1
수수료 등 비용부담 사항
	중도상환수수료 : 중도상환대출금액 × 중도상환수수료율(%) × (대출잔여일수 ÷ 대출기간)
	
☞ 중도상환수수료율은 상품에 따라 0.6%~0.7%입니다. (통상 고정금리 및 혼합금리 방식이 변동금리 방식보다 0.1%p 높습니다.)
	
☞ 최초 대출취급일로부터 3년까지 적용합니다. 대출만기 도래 전 3개월 이내에 상환하는 경우에는 중도상환수수료가 부과되지 
않습니다.
	
※ 중도상환수수료란 대출의 상환기일이 도래하기 전에 대출금을 상환할 경우 고객이 부담하는 금액입니다.
	
• 다만, 기존 대출 계약을 해지하고 동일 은행과 사실상 동일한 계약(기존 